## Proxy

---

> **In one line.** A proxy $P$ implements *exactly* the same interface $I$ as a real object, so the client cannot tell them apart — but on each call it first tests a $\mathrm{condition}$, and either intercepts the call itself or transparently forwards it to the real object.

### 1. The shared interface

Let $I$ be the **shared interface**: the exact set of method signatures that both the real object and the proxy implement. Write the real object as $\mathrm{real}$ and the proxy as $P$. Both realize $I$, and the proxy additionally holds a reference to $\mathrm{real}$ so that it can delegate whenever it chooses to. The decisive fact is that $P$ implements $I$ *exactly* — it exposes no more and no fewer methods than $\mathrm{real}$ — so from the client's vantage point $P$ is **indistinguishable** from the real object:

$$\boxed{\,P : I \longrightarrow I\,} \qquad \text{(same interface as the real object)}.$$

The proxy is thus a map from the interface *to itself*: requests written in $I$ go in, responses consistent with $I$ come out. This type identity is precisely what lets the proxy be slipped in front of $\mathrm{real}$ without the client ever noticing.

### 2. What the proxy does on a call

Let $x$ be a specific method call or request arriving at the proxy — something drawn from the interface $I$. Two things can happen to it. The proxy can perform $\mathrm{intercept}(x)$, its own behaviour *instead of* delegating, such as returning a cached result, raising a permission error, or logging the call. Or it can simply hand the call straight through to $\mathrm{real}(x)$. Which branch fires is governed by a single predicate, the $\mathrm{condition}$ — for instance "is $x$ cached?", "does the user have permission?", or "should this call be logged?":

$$\boxed{\,P(x) = \begin{cases} \mathrm{intercept}(x) & \text{if } \mathrm{condition} \text{ holds} \\[4pt] \mathrm{real}(x) & \text{otherwise} \end{cases}}$$

So every request flows through the same decision chain — arrive, test the predicate, then branch:

$$\underbrace{x}_{\in\, I} \;\xrightarrow{\;\mathrm{condition}?\;}\; \begin{cases} \mathrm{intercept}(x) & \text{true} \\ \mathrm{real}(x) & \text{false} \end{cases}$$

When the $\mathrm{condition}$ is false the proxy is **transparent**: $P(x) = \mathrm{real}(x)$, exactly as if the client had called the real object directly.

### 3. Key conditions

1. **Interface identity** — the proxy's type is *identical* to the real object's type, $P : I \to I$. The client never knows it is talking to a proxy rather than to $\mathrm{real}$.
   $$P : I \to I \;=\; \mathrm{type}(\mathrm{real})$$
2. **Conditional interception** — the proxy intercepts *only* when the $\mathrm{condition}$ holds; otherwise it transparently passes the call through to $\mathrm{real}(x)$.
   $$\mathrm{condition}\ \text{false} \;\Rightarrow\; P(x) = \mathrm{real}(x)$$
3. **Three proxy types** — the same $P : I \to I$ structure specializes into **Virtual** (cache results), **Protection** (check permissions), and **Logging** (record access). Only the choice of $\mathrm{condition}$ and $\mathrm{intercept}$ changes; the interface-preserving shape does not.

&nbsp;

> 🪪 Think of a hotel receptionist. You want room access (the real object $\mathrm{real}$). The receptionist (the proxy $P$) checks your key card (the $\mathrm{condition}$). Valid → opens the room ($\mathrm{real}(x)$). Invalid → access denied ($\mathrm{intercept}(x)$). You interact with the receptionist exactly as you would with direct room access — that sameness is the interface $I$.

### Exercise 7 — Caching Proxy (Virtual)

---

**Scenario:** A `WeatherService` has `get_weather(city)` making slow network calls. The same city is often requested. The proxy intercepts repeated calls and returns cached results.

**Your task:** Write `CachingProxy` implementing the same interface $I$ as `WeatherService`. Condition: "is this city in cache?" Intercept: return the cache. Otherwise: call the real service.

```python
proxy = CachingProxy(WeatherService())
proxy.get_weather("London")   # miss -> real call (slow)
proxy.get_weather("London")   # hit  -> intercept(x), no network
```

**Hints**

- The proxy implements `get_weather(city)` — the *same* method as the real service. This is $P : I \to I$, the interface identity condition.
- Store `self._cache = {}`. Condition: `city in self._cache`. Intercept: `return self._cache[city]`. Otherwise: call real, store the result, return it.

In [ ]:
# --------------------------------
# Real object (implements I) — you do not change this

class WeatherService:
    def get_weather(self, city):                 # the interface I
        print(f"WeatherService: SLOW network call for '{city}'")
        return f"{city}: 18C, cloudy"

# --------------------------------
# Proxy (P : I -> I) — your task: same get_weather(city), but cache results

class CachingProxy:
    def __init__(self, service):
        self._service = service                  # reference to real(x)
        self._cache = {}

    def get_weather(self, city):                 # same signature as I
        # condition: is city cached?  -> intercept(x): return cache
        # otherwise: call real, store, return
        ...

# --------------------------------
proxy = CachingProxy(WeatherService())
print(proxy.get_weather("London"))   # miss -> slow
print(proxy.get_weather("London"))   # hit  -> cached, no network

### Exercise 8 — Protection Proxy

---

**Scenario:** A `Database` has `read(query)` and `write(query)`. Some users are read-only. The proxy checks the user's role before allowing writes.

**Your task:** Write `ProtectionProxy(db, role)`. Condition for `write`: $\mathrm{role} = \texttt{"admin"}$. Intercept: raise `PermissionError`.

```python
admin = ProtectionProxy(Database(), role="admin")
admin.write("INSERT ...")     # allowed -> real(x)

guest = ProtectionProxy(Database(), role="guest")
guest.read("SELECT ...")      # always delegates
guest.write("INSERT ...")     # condition fails -> intercept(x): PermissionError
```

**Hints**

- `read()` always delegates to $\mathrm{real}(x)$ — no interception needed. `write()` checks the condition first.
- The condition is $\mathrm{role} = \texttt{"admin"}$. When it fails, `raise PermissionError(...)` instead of delegating.

In [ ]:
# --------------------------------
# Real object (implements I) — you do not change this

class Database:
    def read(self, query):                       # part of interface I
        print(f"Database: reading [{query}]")
        return "<rows>"
    def write(self, query):                      # part of interface I
        print(f"Database: writing [{query}]")
        return "<ok>"

# --------------------------------
# Proxy (P : I -> I) — your task: same read/write, but guard write by role

class ProtectionProxy:
    def __init__(self, db, role):
        self._db = db                            # reference to real(x)
        self._role = role

    def read(self, query):                       # always delegates -> real(x)
        ...

    def write(self, query):                      # condition: role == 'admin'
        # if not admin -> intercept(x): raise PermissionError
        # else -> delegate to real(x)
        ...

# --------------------------------
admin = ProtectionProxy(Database(), role="admin")
admin.write("INSERT INTO users ...")
print("--------")
guest = ProtectionProxy(Database(), role="guest")
guest.read("SELECT * FROM users")
try:
    guest.write("DELETE FROM users")
except PermissionError as e:
    print(f"Denied: {e}")